# SentinelAI — Notebook 06: CrossEncoder Reranker

**Goal:** Take the Top-K candidates from hybrid search and apply a CrossEncoder model to produce a more precise relevance ranking before passing results to the LLM.

**Where this fits in the pipeline:**
```
MAUDE narrative
      |
      v
HybridSearcher          (Notebook 05)
  BM25 + Vector --> Top-20 candidates
      |
      v
CrossEncoder Reranker   (this notebook)
  scores each (query, PT) pair --> Top-5 candidates
      |
      v
LLM (Ollama llama3.2)   (Notebook 07)
  selects final PT + confidence score
```

**Why a reranker?**  
The hybrid search uses a **bi-encoder** architecture: query and PT name are encoded *separately*, and similarity is measured by vector distance. This is fast and scalable (pre-compute PT embeddings once), but misses subtle relevance signals because the query and candidate never "see" each other during encoding.

A **CrossEncoder** solves this by encoding both together — it can capture complex interactions like negation, context, and clinical specificity. The trade-off: it cannot pre-compute anything, so it's only practical on a small candidate set (20 PTs, not 27,000).

---
## 📖 How to Read This Notebook (Beginner Guide)

This is **Notebook 06** — the reranking stage. Its job is to take the 20 rough
candidates from hybrid search and rank them more precisely before handing them
to the LLM.

### Key concepts explained here

**Why reranking?**
The hybrid search (BM25 + vector) retrieves candidates efficiently, but the
ranking is based on approximate signals — trigram overlap and cosine similarity
of pre-computed vectors. These signals do not actually read the query and the
candidate *together*. A reranker fixes this.

**Bi-encoder vs CrossEncoder** (the fundamental trade-off):

| Property | Bi-encoder (PubMedBERT) | CrossEncoder (MiniLM) |
|---|---|---|
| How it works | Encodes query and PT separately | Encodes (query, PT) pair jointly |
| Speed | Fast — PT vectors are pre-computed | Slow — must run fresh per query |
| Accuracy | Good for retrieval | Much better for ranking |
| Can see across both texts? | No | Yes (full cross-attention) |

This is why we use the bi-encoder for Stage 1 (over 27k PTs) and the CrossEncoder
for Stage 2 (over only 20 candidates).

**CrossEncoder output — logits, not probabilities**
The CrossEncoder outputs a raw "relevance logit" — a number with no fixed range
(can be negative, can be > 10). Higher = more relevant, but you cannot
directly interpret it as a probability. In coding.py, we apply `sigmoid()` to
convert it to a 0-1 value for the final confidence formula.

**Rank change analysis**
We look at how much the CrossEncoder changes the order produced by hybrid search.
Large rank changes mean hybrid search had the right candidates but in the wrong order.
Small changes mean hybrid search was already good.

**Score gap = confidence signal**
If the top-ranked candidate has a CrossEncoder score of 8.2 and the second has 2.1,
the gap (6.1) suggests the top result is clearly better. A small gap (8.2 vs 7.9)
suggests uncertainty — the LLM will need to make a finer decision.

### What this notebook validates
- CrossEncoder correctly identifies the most relevant PT even when hybrid search ranked it lower
- Benchmarks latency: ~20ms per batch of 20 candidates (fast enough for production)
- Validates the `top_k=20 -> top_k=5` funnel design
---

---
## 1. Key Concepts

### Bi-Encoder vs. CrossEncoder

| Property | Bi-Encoder (PubMedBERT) | CrossEncoder (MiniLM) |
|----------|------------------------|----------------------|
| Input | query → vector, PT → vector (separately) | [query, PT] → score (jointly) |
| Speed | Fast at query time (dot product) | Slow (full forward pass per pair) |
| Scalability | Can pre-compute PT embeddings | Must run per query per candidate |
| Accuracy | Good | Better — joint attention sees interactions |
| Use case | Initial retrieval (27k PTs) | Reranking (Top-20 candidates) |

### MiniLM — the model

**MiniLM** (Miniaturized Language Model) is a *knowledge-distilled* version of BERT:  
- 6 transformer layers (vs. 12 in BERT-base)
- 22MB model size (vs. 440MB for PubMedBERT)
- ~4x faster inference than BERT-base
- Trained on **MS MARCO** — a dataset of 530,000 real search queries with human-judged relevance labels

**Knowledge distillation** = a smaller *student* model learns to mimic the output distribution of a larger *teacher* model. The student is faster but retains most of the teacher's accuracy.

### Why ms-marco-MiniLM-L-6-v2 for medical text?

MS MARCO is a general web-search dataset — not medical. But CrossEncoders show strong zero-shot transfer to domain-specific relevance tasks because the task ("is this passage relevant to this query?") is universal. For our use case (matching a clinical description to a standardized term), the model generalizes well.

A domain-specific alternative would be `cross-encoder/nli-MiniLM2-L6-H768`, but ms-marco-MiniLM tends to outperform on asymmetric relevance tasks (short PT name vs. longer narrative).

### CrossEncoder output: logits, not probabilities

The CrossEncoder outputs a raw **logit** (unbounded real number) — not a probability.  
- Higher logit = more relevant
- The absolute values are not interpretable — only the relative ranking matters
- We can optionally apply sigmoid to get a (0,1) range, but we don't need to for ranking

In [ ]:
import sys
sys.path.insert(0, '..')

import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from dotenv import load_dotenv
import psycopg2

load_dotenv()

def get_db_url():
    url = os.getenv('DATABASE_URL')
    if url:
        return url
    host = os.getenv('POSTGRES_HOST', 'localhost')
    port = os.getenv('POSTGRES_PORT', '5432')
    db   = os.getenv('POSTGRES_DB', 'vigilex')
    user = os.getenv('POSTGRES_USER', 'vigilex')
    pw   = os.getenv('POSTGRES_PASSWORD', '')
    return f'postgresql://{user}:{pw}@{host}:{port}/{db}'

conn = psycopg2.connect(get_db_url())
print('DB connected.')

---
## 2. Load Models and Searcher

In [ ]:
from src.vigilex.coding.hybrid_search import HybridSearcher, EmbeddingModel
from src.vigilex.coding.reranker import CrossEncoderReranker

# Load PubMedBERT for hybrid search
print('Loading PubMedBERT...')
embed_model = EmbeddingModel()
searcher = HybridSearcher(conn, embedding_model=embed_model, candidate_pool=100)

# Load CrossEncoder -- much smaller, loads in seconds
print('Loading CrossEncoder (MiniLM-L-6)...')
reranker = CrossEncoderReranker()

print('All models loaded.')

---
## 3. Run the Full Two-Stage Pipeline

In [ ]:
TEST_QUERIES = [
    ("Clear clinical term",
     "Patient experienced hypoglycaemia after insulin bolus delivery"),

    ("Consumer language",
     "blood sugar dropped very low, patient was shaking and confused"),

    ("Device malfunction",
     "insulin pump stopped delivering insulin, occlusion alarm triggered"),

    ("Skin reaction",
     "redness and swelling at the infusion site, possible allergic reaction"),

    ("CGM inaccuracy",
     "continuous glucose monitor showed incorrect readings, sensor failure"),

    ("Multi-symptom",
     "patient reported nausea, dizziness and loss of consciousness"),
]

pipeline_results = {}

for label, query in TEST_QUERIES:
    # Stage 1: Hybrid retrieval -- get top 20 candidates
    candidates = searcher.search(query, top_k=20)

    # Stage 2: CrossEncoder reranking -- score each (query, PT) pair
    reranked = reranker.rerank(query, candidates, top_k=5)

    pipeline_results[label] = {
        'query': query,
        'candidates': candidates,
        'reranked': reranked
    }

    print(f'\n=== {label} ===')
    print(f'Query: "{query}"')
    print(f'{'PT Name':<50} {'CE Score':>10} {'RRF Rank':>10}')
    print('-' * 72)
    for r in reranked:
        print(f'{r.pt_name:<50} {r.crossencoder_score:>10.3f} {r.rrf_rank:>10}')

---
## 4. Rank Change Analysis

How much does the reranker change the order compared to hybrid search?  
A large rank change means the CrossEncoder found a better answer that the bi-encoder missed.

In [ ]:
for label, data in pipeline_results.items():
    query = data['query']
    reranked = data['reranked']

    print(f'\n=== {label} ===')
    print(f'Query: "{query[:70]}"')
    print(f'{'PT Name':<45} {'RRF rank':>9} {'CE rank':>8} {'Change':>8}')
    print('-' * 72)

    for ce_rank, r in enumerate(reranked, start=1):
        change = r.rrf_rank - ce_rank  # positive = moved UP after reranking
        arrow = '↑' * change if change > 0 else ('↓' * abs(change) if change < 0 else '=')
        print(f'{r.pt_name:<45} {r.rrf_rank:>9} {ce_rank:>8} {arrow:>8}')

---
## 5. Score Distribution

Visualise the CrossEncoder scores across all candidates for one query.  
A large score gap between rank 1 and rank 2 = the model is confident. A small gap = uncertain.

In [ ]:
def plot_ce_scores(label, query, candidates, reranked, n_show=20):
    # Score all candidates (not just top 5) for the full distribution
    all_pairs = [(query, c.pt_name) for c in candidates[:n_show]]
    all_scores = reranker.model.predict(all_pairs)

    pt_names = [c.pt_name[:45] for c in candidates[:n_show]]
    colors = ['coral' if i < 5 else 'steelblue' for i in range(n_show)]

    # Sort by score for plot
    sorted_pairs = sorted(zip(all_scores, pt_names, colors), reverse=True)
    scores_s, names_s, colors_s = zip(*sorted_pairs)

    fig, ax = plt.subplots(figsize=(12, 6))
    bars = ax.barh(range(len(names_s)), scores_s, color=colors_s, alpha=0.85)
    ax.set_yticks(range(len(names_s)))
    ax.set_yticklabels(names_s, fontsize=8)
    ax.set_xlabel('CrossEncoder score (logit)')
    ax.set_title(f'{label}\n"{query[:80]}"', fontsize=10)

    coral_patch = mpatches.Patch(color='coral',     label='Top 5 (pass to LLM)')
    blue_patch  = mpatches.Patch(color='steelblue', label='Remaining candidates')
    ax.legend(handles=[coral_patch, blue_patch], loc='lower right')

    plt.tight_layout()
    plt.show()

import matplotlib.patches as mpatches

for label, data in pipeline_results.items():
    plot_ce_scores(
        label,
        data['query'],
        data['candidates'],
        data['reranked']
    )

---
## 6. Confidence Signal: Score Gap

The gap between rank 1 and rank 2 CrossEncoder scores is a proxy for coding confidence.  
We will use this later to decide whether to pass the result directly to the LLM or flag it for human review.

- Large gap (> 2.0) → model is confident → low ambiguity
- Small gap (< 0.5) → ambiguous → LLM needs more context

In [ ]:
print(f'{'Query':<30} {'Rank1 PT':<45} {'Gap (rank1-rank2)':>18}')
print('-' * 95)

for label, data in pipeline_results.items():
    r = data['reranked']
    if len(r) >= 2:
        gap = r[0].crossencoder_score - r[1].crossencoder_score
        confidence = 'HIGH' if gap > 2.0 else ('MED' if gap > 0.5 else 'LOW')
        print(f'{label:<30} {r[0].pt_name:<45} {gap:>10.3f}  [{confidence}]')

---
## 7. Performance Benchmark

CrossEncoder latency matters for the coding worker throughput.

In [ ]:
import time

BENCHMARK_QUERY = "patient experienced hypoglycaemia after insulin bolus delivery"
N_RUNS = 10
CANDIDATE_SIZES = [5, 10, 20]

# Get candidates once
candidates = searcher.search(BENCHMARK_QUERY, top_k=20)

print(f'CrossEncoder latency vs. candidate set size:')
print(f'{'Candidates':>12} {'Mean (ms)':>12} {'Min (ms)':>10} {'Max (ms)':>10}')
print('-' * 46)

for n in CANDIDATE_SIZES:
    times = []
    for _ in range(N_RUNS):
        t0 = time.time()
        reranker.rerank(BENCHMARK_QUERY, candidates[:n], top_k=5)
        times.append((time.time() - t0) * 1000)
    print(f'{n:>12} {sum(times)/len(times):>12.1f} {min(times):>10.1f} {max(times):>10.1f}')

print()
print('Note: CrossEncoder is fast enough for real-time use.')
print('Full pipeline (hybrid search + reranker): measure total below.')

In [ ]:
# Full pipeline benchmark
N_RUNS = 5
times = []

for _ in range(N_RUNS):
    t0 = time.time()
    cands = searcher.search(BENCHMARK_QUERY, top_k=20)
    reranker.rerank(BENCHMARK_QUERY, cands, top_k=5)
    times.append((time.time() - t0) * 1000)

print(f'Full pipeline (hybrid + reranker), {N_RUNS} runs:')
print(f'  Mean: {sum(times)/len(times):.0f} ms')
print(f'  Min:  {min(times):.0f} ms')
print(f'  Max:  {max(times):.0f} ms')
print()
print('Bottleneck: PubMedBERT query encoding (~80% of time).')
print('CrossEncoder adds only marginal overhead on top of hybrid search.')

---
## 8. Summary & Next Steps

**What we built:**
- `reranker.py` — `CrossEncoderReranker` class wrapping `cross-encoder/ms-marco-MiniLM-L-6-v2`
- Two-stage pipeline: HybridSearcher (Top-20) → CrossEncoder (Top-5)
- Score gap as a confidence proxy for downstream LLM processing

**Technical summary of the full retrieval stack so far:**

| Stage | Method | Model | Output |
|-------|--------|-------|--------|
| 1a. Lexical retrieval | BM25 via pg_trgm | PostgreSQL built-in | Top-100 by trigram sim |
| 1b. Semantic retrieval | ANN via pgvector | PubMedBERT (768d) | Top-100 by cosine sim |
| 1c. Fusion | Weighted RRF | — | Top-20 fused candidates |
| 2. Reranking | CrossEncoder | MiniLM-L-6 | Top-5 by relevance logit |
| 3. Final coding | LLM | llama3.2 (Ollama) | 1 PT + confidence score |

**Next step — Notebook 07:** LLM-based final coding with Ollama (llama3.2).  
The LLM receives the Top-5 candidates from the reranker and selects the single best PT with a structured JSON output including a confidence score and brief justification.